<a href="https://colab.research.google.com/github/masha713/ML_demeshok/blob/main/%D0%9B%D0%A0_9_%D0%94%D0%B5%D0%BC%D0%B5%D1%88%D0%BE%D0%BA_3_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студентки:** Демешок Марія Віталіївна  
**Група:** 3-10  
**Дата:** 04.11.26

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV, cross_validate
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [5]:
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [6]:
unique_users = ratings_df['user_id'].nunique()
unique_movies = ratings_df['item_id'].nunique()

min_rating = ratings_df['rating'].min()
max_rating = ratings_df['rating'].max()
mean_rating = ratings_df['rating'].mean()
missing_values = ratings_df.isnull().sum()

print(f"Кількість унікальних користувачів: {unique_users}")
print(f"Кількість унікальних фільмів: {unique_movies}")
print(f"Рейтинг: мін = {min_rating}, макс = {max_rating}, середнє = {mean_rating:.2f}")
print("\nПропущені значення в розрізі стовбців:")
print(missing_values)

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Рейтинг: мін = 1.0, макс = 5.0, середнє = 3.53

Пропущені значення в розрізі стовбців:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

_Набір даних є повним і містить 100 000 оцінок без жодних пропусків, які розподілені між 943 унікальними користувачами та 1682 фільмами. Рейтинги виставлялися за 5-бальною шкалою, а середнє значення 3.53 вказує на те, що користувачі схильні давати фільмам переважно помірно позитивні оцінки.._

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [7]:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
  )

print('Навчальна вибірка:', train_df.shape)
print('Тестова вибірка:', test_df.shape)

Навчальна вибірка: (80000, 4)
Тестова вибірка: (20000, 4)


In [9]:
reader = Reader(rating_scale=data.reader.rating_scale)
train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(
    test_df[['user_id', 'item_id', 'rating']]
    .itertuples(index=False, name=None)
)

full_trainset = train_data.build_full_trainset()
print(f"Розмір навчальної вибірки (train): {full_trainset.n_ratings} оцінок")
print(f"Розмір тестової вибірки (testset): {len(testset)} кортежів")

Розмір навчальної вибірки (train): 80000 оцінок
Розмір тестової вибірки (testset): 20000 кортежів


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [13]:
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Для моделей SVD та KNN обрано найбільш критичні параметри, які безпосередньо впливають на точність рекомендацій та швидкість навчання пристрою. У сітках параметрів варіюються розмірність прихованих факторів і рівень регуляризації (для SVD), а також кількість сусідів, метрики подібності та напрямок порівняння користувачів чи товарів (для KNN), що дозволяє знайти оптимальний баланс між якістю прогнозів та захистом моделі від перенавчання.._

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [18]:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_svd.fit(train_data)

In [19]:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_knn.fit(train_data)

In [20]:
print('Найкращий результат SVD:')
print('RMSE:', grid_search_svd.best_score['rmse'])
print('MAE:', grid_search_svd.best_score['mae'])
print('Параметри за RMSE:', grid_search_svd.best_params['rmse'])

print('Найкращий результат KNNBasic:')
print('RMSE:', grid_search_knn.best_score['rmse'])
print('MAE:', grid_search_knn.best_score['mae'])
print('Параметри за RMSE:', grid_search_knn.best_params['rmse'])

best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

Найкращий результат SVD:
RMSE: 0.9417749231920144
MAE: 0.7425276653554779
Параметри за RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Найкращий результат KNNBasic:
RMSE: 1.0297278489639847
MAE: 0.8163947396854526
Параметри за RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.9418 | 0.7425 | {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05} |
| KNNBasic | 1.0297 | 0.8164 | {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}} |
                                                                                
_Кросвалідація показала, що алгоритм SVD впорався із завданням значно краще, ніж KNNBasic у нього помітно менше помилок (RMSE 0.941 проти 1.029). Для моделі SVD найвлучнішим виявився вибір середньої кількості факторів (50) та максимальний захист від перенавчання. У моделі KNNBasic найкраще спрацював варіант, коли система шукає схожість саме між самими користувачами (user-based) серед 40 найближчих_

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [22]:
trainset = train_data.build_full_trainset()

best_svd = SVD(
    **best_svd_params,
    random_state=42
    )
best_svd.fit(trainset)

best_knn = KNNBasic(
    **best_knn_params
    )
best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [26]:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

print('\nРезультати SVD на тестовій вибірці:')
svd_rmse = accuracy.rmse(
    svd_predictions,
    verbose=True
    )
svd_mae = accuracy.mae(
    svd_predictions,
    verbose=True
    )
print('\nРезультати KNNBasic на тестовій вибірці:')
knn_rmse = accuracy.rmse(
    knn_predictions,
    verbose=True)
knn_mae = accuracy.mae(
    knn_predictions,
    verbose=True)


Результати SVD на тестовій вибірці:
RMSE: 0.9192
MAE:  0.7234

Результати KNNBasic на тестовій вибірці:
RMSE: 1.0120
MAE:  0.7984


In [27]:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
}).sort_values('RMSE')

print('\nПорівняння моделей:')
display(results)

best_model_name = results.iloc[0]['Model']
print(f"Найкраща модель за RMSE: {best_model_name}")


Порівняння моделей:


,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


Найкраща модель за RMSE: SVD


**Порівняння з CV:** _Оцінки помилок на кросвалідації та на незалежній тестовій вибірці відрізняються несуттєво, що свідчить про стабільність моделей та відсутність перенавчання (overfitting). Для алгоритму SVD тестовий показник RMSE навіть виявився трохи кращим (0.919 проти 0.941 на CV), оскільки фінальне навчання проходило на повному обсязі тренувальних даних. У моделі KNNBasic результати залишилися практично на тому ж рівні (тестовий RMSE 1.011 проти 1.029 на CV), підтверджуючи, що обидві моделі добре генералізують нові дані.._

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [28]:
user_id = '196'

if results.iloc[0]['Model'] == 'SVD':
    best_model = best_svd
else:
    best_model = best_knn

print(f"Для рекомендацій використовується найкраща модель: {results.iloc[0]['Model']}")

try:
    inner_user_id = trainset.to_inner_uid(user_id)

    user_ratings = trainset.ur[inner_user_id]
    rated_inner_items = set([item_id for (item_id, _) in user_ratings])

    all_inner_items = set(trainset.all_items())
    unrated_inner_items = all_inner_items - rated_inner_items

    recommendations = []
    for inner_item_id in unrated_inner_items:
        raw_item_id = trainset.to_raw_iid(inner_item_id)
        predicted_rating = best_model.predict(uid=user_id, iid=raw_item_id).est

        recommendations.append({
            'item_id': raw_item_id,
            'predicted_rating': predicted_rating
        })

    recommendations_df = pd.DataFrame(recommendations).sort_values(
        by='predicted_rating',
        ascending=False
    ).head(10)

    print(f"\nТоп-10 рекомендацій для користувача {user_id}:")
    display(recommendations_df)

except ValueError:
    print(f"\nПомилка: Користувача з ID '{user_id}' немає в навчальній вибірці (trainset).")

Для рекомендацій використовується найкраща модель: SVD

Топ-10 рекомендацій для користувача 196:


,item_id,predicted_rating
1544,320,4.490358
153,64,4.470680
231,480,4.413710
270,313,4.401920
431,513,4.399624
74,483,4.398092
92,318,4.392102
139,656,4.385379
357,22,4.377388
12,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?

_Для фінальних рекомендацій я обрала модель SVD. На етапі кросвалідації вона виявилася точнішою за KNNBasic, показавши меншу помилку RMSE 0.9418 проти 1.0297._

2. Які найкращі гіперпараметри отримано для обох моделей?

_Для SVD підійшли параметри: n_factors: 50, lr_all: 0.01, reg_all: 0.05, а для KNNBasic найкращими стали: k: 40, min_k: 1, метрика cosine та режим user_based: True._

3. Які підсумкові значення RMSE та MAE на тестовій вибірці?

_На незалежній тестовій вибірці SVD видала RMSE 0.9191 та MAE 0.7233, а KNNBasic RMSE 1.0119 та MAE 0.7984._

4. Чи є різниця між результатами CV та тестування?

_Особливої різниці немає, метрики майже однакові. На тесті результати навіть трохи покращилися (наприклад, у SVD помилка впала з 0.9418 до 0.9191). Це логічно, адже перед тестом ми донавчили моделі на всій доступній тренувальній вибірці._

5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

_Якщо на сайт прийде зовсім новий юзер або додадуть новий фільм без оцінок, система не зможе зробити адекватний прогноз, бо їй просто немає з чим порівнювати._


**Висновок:** _Під час виконання лабораторної роботи я розібралася, як працюють рекомендаційні системи, та протестувала два алгоритми на датасеті MovieLens-100k. За всіма метриками переміг алгоритм матричної факторизації SVD його помилка RMSE на тесті склала всього 0.9191, тоді як у базового сусідського методу KNNBasic вона перевищила одиницю (1.0119). Це показує, що приховані фактори у SVD набагато краще знаходять закономірності в уподобаннях людей. За допомогою найкращої моделі я згенерувала Топ-10 рекомендацій для користувача під номером 196. Туди потрапили лише ті фільми, які він ще не бачив, а прогнозовані оцінки вийшли дуже високими: від 4.37 до 4.49 бала. Оскільки результати кросвалідації та фінального тесту збігаються, можна впевнено сказати, що підбір параметрів через GridSearchCV пройшов успішно, і модель не перенавчилася._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.